# GX 02
# The Bessel Beam

In [1]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# -----------------------------
# Simulation parameters
# -----------------------------
N = 512                  # grid size
L = 5e-3                 # physical size (5 mm)
wavelength = 633e-9      # wavelength (633 nm)
k0 = 2*np.pi / wavelength

z_max = 0.5

dx = L / N

# Real space grid
x = np.linspace(-L/2, L/2-dx, N)
y = np.linspace(-L/2, L/2-dx, N)
X, Y = np.meshgrid(x, y)

# Frequency space grid (fftshifted)
fx = np.fft.fftshift(np.fft.fftfreq(N, d=dx))
fy = np.fft.fftshift(np.fft.fftfreq(N, d=dx))
FX, FY = np.meshgrid(fx, fy)

KX = 2*np.pi * FX
KY = 2*np.pi * FY
K_R = np.sqrt(KX**2 + KY**2)

In [2]:
# - No modification necessary -

def angular_spectrum_propagation(field, z):
    F = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(field)))

    kz = np.sqrt(np.maximum(0, k0**2 - KX**2 - KY**2))
    H = np.exp(1j * kz * z)

    F_prop = F * H
    field_prop = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(F_prop)))

    return field_prop

# Part 1
Let's begin by creating a bessel beam from scratch.

Recall that a bessel beam is created from the circle of all plane waves sharing a common $k_z$. This means that it is natural to define the bessel beam first in the frequency space and then convert it back to real space before propagation.

Start by implementing the function bessel_ring, which takes a radius and thickness as arguments. Both the radius and thickness provided are in units of k-space (i.e. units of 1/m). The ring will be of equal amplitude and phase at all locations, and the ring's thickness should be centered about the radius value. A frequency space meshgrid has already been provided for you that you should use to create the bessel beam. Note that what has been provided uses an fftshift to place the frequency coordinates (0,0) at the center of the grid for clarity, but traditionally in frequency space (0,0) is located at the top left of the image.

After generating the bessel beam, use the provided plotting code to examine the beam in frequency and real space at z=0, as well as the propagation of the beam in z. Discuss the following questions:
1)  What do you observe as the effect of changing the radius and thickness of the ring in the real space?
2)  What effect do each of these parameters have on how far the beam is able to propagate?
3)  Do your observations for beam propagation match what you would expect from theory?

In [14]:
def bessel_ring(radius, thickness):
    """
    Create a ring in k-space.

    Parameters:
        radius (float): radial spatial frequency (rad/m)
        thickness (float): thickness of ring (rad/m)

    Returns:
        2D numpy array representing ring in k-space
    """

    # ============================
    # STUDENT CODE HERE
    # ============================

    ring = np.logical_and(K_R >= radius - thickness/2,
                          K_R <= radius + thickness/2)

    return ring.astype(float)

In [4]:
# - No modification necessary -

def interactive_bessel(ring_radius,
                       ring_thickness):

    # Generate ring
    bessel_k = bessel_ring(ring_radius, ring_thickness)

    # Convert to real space
    field = np.fft.fftshift(
                np.fft.ifft2(
                    np.fft.ifftshift(bessel_k)
                )
            )

    intensity = np.abs(field)**2

    # Propagation
    z_steps = 50
    z_vals = np.linspace(0, z_max, z_steps)

    xz_slice = np.zeros((z_steps, N))

    for i, z in enumerate(z_vals):
        field_z = angular_spectrum_propagation(field, z)
        xz_slice[i,:] = np.abs(field_z[N//2, :])**2

    # Plot
    fig, axs = plt.subplots(1, 3, figsize=(18,5))

    # k-space
    im0 = axs[0].imshow(bessel_k,
                        extent=[fx[0], fx[-1], fy[0], fy[-1]],
                        cmap='inferno')
    axs[0].set_title("k-Space Ring")
    plt.colorbar(im0, ax=axs[0])

    # Real space
    im1 = axs[1].imshow(intensity,
                        extent=[x[0]*1e3,x[-1]*1e3,
                                y[0]*1e3,y[-1]*1e3])
    axs[1].set_title("Real Space Intensity")
    plt.colorbar(im1, ax=axs[1])

    # Propagation
    im2 = axs[2].imshow(xz_slice,
                        extent=[x[0]*1e3,x[-1]*1e3,
                                z_max*100,0],
                        aspect='auto')
    axs[2].set_title("Propagation (x-z)")
    axs[2].set_xlabel("x (mm)")
    axs[2].set_ylabel("z (cm)")
    plt.colorbar(im2, ax=axs[2])

    plt.tight_layout()
    plt.show()


interact(
    interactive_bessel,
    ring_radius=FloatSlider(value=1e5, min=1e4, max=2e5, step=1e4),
    ring_thickness=FloatSlider(value=5e3, min=1e3, max=1e4, step=1e3)
);

interactive(children=(FloatSlider(value=100000.0, description='ring_radius', max=200000.0, min=10000.0, step=1…

## Discussion
1) Increasing the radius of the ring appears to scale the beam down in real space. Decreasing the thickness of the ring makes more of the fringes visible, though this is hard to visualize for a large ring radius.
2) Increasing the radius of the ring makes the beam propagate less far before it dissipates. The same effect occurs when the ring thickness is increased.
3) Yes, these observations match what we would expect to see. When the radius of the ring is increased, we expect to observe that our finite plane waves diverge more quickly and therefore our interference pattern should break down earlier. When the thickness of the ring is increased, more frequencies are included in our input field that do not have quite the same $k_z$. Because this deviation is small, we expect to see that the behavior of our bessel beam only breaks down after the plane waves have propagated enough for the deviations to become visible. For larger thicknesses, we anticipate a larger deviation and thus a faster breakdown.

# Part 2
Now that we can create a bessel beam, let's investigate the effect of applying a square aperture to a bessel beam.

Using the bessel_ring function that you have already generated, implement the aperture_bessel function. This function will take an aperture_radius parameter in addition to the previous ring_radius and ring_thickness parameters. First, you should create a bessel beam in k-space. After this, it must be converted to real space (look at the existing plotting code for how to do this). Once it is in real space, you should apply a square aperture that has a half-width (radius) equal to the passed parameter. 

After generating the filtered bessel beam, use the provided plotting code to examine the beam in real space at z=0, as well as the propagation of the beam in z. Discuss the following questions:
1)  What is the main trend you observe between aperture size and propagation distance?
2)  With this in mind, how should I adjust my simulation domain if the bessel beam I am simulating does not propagate as far as I expect it to?

In [12]:
def aperture_bessel(aperture_radius,
                    ring_radius,
                    ring_thickness):
    """
    Generate Bessel beam and apply rectangular aperture.
    """

    # ============================
    # STUDENT CODE HERE
    # ============================

    # Create Bessel beam in k-space
    bessel_k = bessel_ring(ring_radius, ring_thickness)

    # Convert to real space
    field = np.fft.fftshift(
                np.fft.ifft2(
                    np.fft.ifftshift(bessel_k)
                )
            )

    # Apply circular aperture
    aperture = np.logical_and(np.abs(X) <= aperture_radius, np.abs(Y) <= aperture_radius)
    field_ap = field * aperture

    return field_ap

In [13]:
# - No modification necessary -

def interactive_aperture(ring_radius,
                         ring_thickness,
                         aperture_radius):

    field_ap = aperture_bessel(aperture_radius,
                               ring_radius,
                               ring_thickness)

    intensity_ap = np.abs(field_ap)**2

    # Propagation
    z_steps = 50
    z_vals = np.linspace(0, z_max, z_steps)

    xz_slice = np.zeros((z_steps, N))

    for i, z in enumerate(z_vals):
        field_z = angular_spectrum_propagation(field_ap, z)
        xz_slice[i,:] = np.abs(field_z[N//2, :])**2

    # Plot
    fig, axs = plt.subplots(1, 2, figsize=(14,5))

    # Real space after aperture
    im0 = axs[0].imshow(intensity_ap,
                        extent=[x[0]*1e3,x[-1]*1e3,
                                y[0]*1e3,y[-1]*1e3])
    axs[0].set_title("Real Space (With Aperture)")
    axs[0].set_xlabel("x (mm)")
    axs[0].set_ylabel("y (mm)")
    plt.colorbar(im0, ax=axs[0])

    # Propagation
    im1 = axs[1].imshow(xz_slice,
                        extent=[x[0]*1e3,x[-1]*1e3,
                                z_max*100,0],
                        aspect='auto')
    axs[1].set_title("Propagation (x-z)")
    axs[1].set_xlabel("x (mm)")
    axs[1].set_ylabel("z (cm)")
    plt.colorbar(im1, ax=axs[1])

    plt.tight_layout()
    plt.show()


interact(
    interactive_aperture,
    ring_radius=FloatSlider(value=1e5, min=1e4, max=2e5, step=1e4),
    ring_thickness=FloatSlider(value=5e3, min=1e3, max=1e4, step=1e3),
    aperture_radius=FloatSlider(value=0.5e-3, min=0, max=4e-3, step=0.1e-3, readout_format='.4f')
);

interactive(children=(FloatSlider(value=100000.0, description='ring_radius', max=200000.0, min=10000.0, step=1…

# Bonus
How should we think about the effect of applying an aperture to our system in frequency space? What would this do to the initial frequency ring that we have created, and how can we predict the result of applying an aperture from this?

A multiplication in real space (our beam times a square aperture) is the same as a convolution in frequency space. We know that the frequency space representation of our bessel beam is a ring, and the frequency space representation of our square aperture is a 2D sinc. This means that in frequency space, our bessel beam after applying an aperture is the convolution of a ring with a sinc function, or more simply put, a blurred ring. This means that we should expect to see a similar effect to what we observed when we manually increased the thickness of our ring in frequency space, where the slightly deviating frequencies accumulate discrepancies over significant z propagation and ultimately lead to a breakdown of the beam.